# gpt

> ChatGPT web conversations as dialogs

In [ ]:
#| default_exp gpt

Convert ChatGPT web conversations into dialogs.

## Conversation payloads

The web app renders every conversation from a JSON payload, and three sources serve the same node schema: `GET /backend-api/conversation/<id>` (the owner's login), `GET /backend-api/share/<id>` (share links, including workspace `/share/e/` ones), and each conversation in the Settings data export. A node wraps a `message` carrying an `author` role, a typed `content`, a `recipient`, and `metadata`. `convo2dlg` converts any of these payloads; `url2convo` fetches one through a logged-in browser tab.

In [ ]:
#| export
import json, re
from fastcore.utils import *
from fastllm.chat import mk_msg
from aidialog.msg_parts import Msg, Text, Thinking, ToolUse, ToolResult
from aidialog.hist import chat2dlg

In [ ]:
from importlib.resources import files
from fastcore.test import *

In [ ]:
# test data
hugo = json.loads((files('llmsurgery')/'data'/'gpt'/'hugo.json').read_text())
hugo['title'], len(hugo['mapping'])

('2024 Hugo Winner', 11)

## The node list

A share payload arrives pre-flattened in `linear_conversation`. A conversation payload instead carries a `mapping` tree where message edits and retries branch it. The active path hangs from `current_node`. `convo_nodes` normalizes either wrapper, so everything downstream is source-agnostic.

In [ ]:
#| export
def convo_nodes(convo):
    "Conversation nodes in order: `linear_conversation` when present, else the `current_node` parent chain"
    if lin := convo.get('linear_conversation'): return L(lin)
    m,res,nid = convo['mapping'],[],convo.get('current_node')
    while nid:
        res.append(m[nid])
        nid = m[nid].get('parent')
    return L(reversed(res))

`hugo` is the `mapping` form, so `convo_nodes` walks it; wrapping the same nodes as `linear_conversation` takes the pre-flattened branch to the same result.

In [ ]:
nodes = convo_nodes(hugo)
roles = [n['message']['author']['role'] if n.get('message') else None for n in nodes]
test_eq(roles, [None]+(['user']+['assistant']*4)*2)
test_eq(convo_nodes(dict(linear_conversation=nodes)), nodes)
nodes.itemgot('id')

['node10', 'node2', 'node8', 'node3', 'node4', 'node5', 'node1', 'node7', 'node0', 'node9', 'node6']

## Canonical messages

The dialect map, mirroring `recs2chat` in `llmsurgery.ant`: ChatGPT discriminates assistant activity by `content_type` and `recipient` where Claude uses content-block types. A message addressed to a `recipient` other than `'all'` is a tool call, and a `tool`-role message is its result; `_content` reads the payload string from whichever content shape (`parts` or a code `text`), so every tool maps through one line regardless of its name. Results carry no call id, so each pairs with the latest call positionally. Inline citations arrive as marker glyphs whose resolved links sit in `metadata.content_references`; `_splice_refs` swaps each marker for its markdown links, splicing at the ref's recorded offsets and verifying `matched_text` there first—footnote refs carry a plain-space `matched_text` and a zero-width offset, so a text-wide replace would corrupt the message (it deleted every space, in testing). `reasoning_recap` ("Thought for 2s") and `model_editable_context` are bookkeeping; `_is_convo` drops them, along with hidden `system` nodes.

In [ ]:
#| export
def _ref_md(r):
    its = r.get('items') or []
    return ' ('+', '.join(f"[{i.get('title') or i['url']}]({i['url']})" for i in its)+')' if its else ''

def _splice_refs(txt, refs):
    "Replace citation markers in `txt` with markdown links, splicing at each ref's verified offsets"
    for r in sorted(refs or [], key=lambda r: -r['start_idx']):
        s,e = r['start_idx'],r['end_idx']
        if txt[s:e]==r.get('matched_text'): txt = txt[:s]+_ref_md(r)+txt[e:]
    return txt

def _content(msg):
    "A message's payload string, from whichever content shape, with citations spliced in"
    c = msg['content']
    txt = '\n\n'.join(p for p in c['parts'] if isinstance(p,str)) if 'parts' in c else c.get('text','')
    return _splice_refs(txt, (msg.get('metadata') or {}).get('content_references'))

def _is_convo(node):
    "Is `node` part of the conversation proper (vs hidden bookkeeping)?"
    msg = node.get('message')
    if not msg: return False
    if msg['author']['role']=='system' or (msg.get('metadata') or {}).get('is_visually_hidden_from_conversation'): return False
    return msg['content'].get('content_type') not in ('reasoning_recap','model_editable_context')

def nodes2chat(
    nodes, # Conversation nodes, e.g. from `convo_nodes`
):
    "Canonical messages for the conversation in `nodes`"
    msgs,tu = [],None
    for n in filter(_is_convo, nodes):
        msg = n['message']
        role,c = msg['author']['role'],msg['content']
        if role=='user': msgs.append(mk_msg(_content(msg)))
        elif role=='tool':
            kw = dict(id=tu.id,name=tu.name,arguments=tu.arguments) if tu else dict(id=msg['id'],name=msg['author'].get('name'))
            msgs.append(Msg(role='tool', content=[ToolResult(text=_content(msg), **kw)]))
        elif msg.get('recipient','all')!='all':
            tu = ToolUse(id=msg['id'], name=msg['recipient'], arguments=dict(input=_content(msg)))
            msgs.append(Msg(role='assistant', content=[tu]))
        elif c.get('content_type')=='thoughts':
            msgs.append(Msg(role='assistant', content=[Thinking(text='\n\n'.join(t.get('content','') for t in c.get('thoughts',[])))]))
        else: msgs.append(Msg(role='assistant', content=[Text(text=_content(msg))]))
    return msgs

In [ ]:
msgs = nodes2chat(convo_nodes(hugo))
seq = [(m.role, type(m.content[0]).__name__) for m in msgs]
test_eq(seq, [('user','Text'),('assistant','ToolUse'),('assistant','Thinking'),('assistant','Text')]*2)
test_eq(msgs[1].content[0].name, 'web')
rep = first(m.content[0].text for m in msgs if m.role=='assistant' and type(m.content[0]).__name__=='Text')
assert '](http' in rep, 'citation marker was not replaced by a markdown link'
assert 'Best Novel' in rep, 'sources_footnote splice must not eat the surrounding spaces'
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Search the web for which novel won the 2024 Hugo Award for Best Novel, and cite your sources.', citations=None)]),
 Msg(role='assistant', content=[ToolUse(raw=None, cache_control=None, id='node8', name='web', arguments={'input': 'search("Search the web for which novel won the 2024 Hugo Award for Best Novel, and cite your sources.")'}, server=False, text=None)]),
 Msg(role='assistant', content=[Thinking(raw=None, cache_control=None, text='', showthink=False)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='The **2024 Hugo Award for Best Novel** was won by ***Some Desperate Glory*** by **Emily Tesh**.\n\nThe official Hugo Awards voting statistics rank it **#1 in Best Novel**, ahead of *Translation State* by Ann Leckie and *The Adventures of Amina al-Sirafi* by Shannon Chakraborty.  ([2024 HUGO VOTING](https://www.thehugoawards.org/wp-content/uploads/2024/08/2024_hugo_statistics.pdf?utm_source=c

## To a dialog

`convo2dlg` composes the read path like `sess2dlg`: `chat2dlg` over the canonical messages. `nodes2chat` has already dropped the hidden and system bookkeeping (see `_is_convo`), so the dialog holds only the conversation proper—one prompt per user turn.

In [ ]:
#| export
def convo2dlg(
    convo, # Conversation payload dict, or path to a saved .json
    name=None, # Dialog name; the conversation title if None
    mx=2000, # Maximum rendered tool string length
):
    "A ChatGPT conversation as a dialog, one prompt per user turn"
    if isinstance(convo,(str,Path)): convo = json.loads(Path(convo).read_text())
    return chat2dlg(nodes2chat(convo_nodes(convo)), name or convo.get('title') or 'conversation', mx=mx)

In [ ]:
d = convo2dlg(hugo)
test_eq([m.msg_type for m in d.messages], ['prompt','prompt'])
d.view(incl_out=True)

AttributeError: 'Dialog' object has no attribute 'view'

## Fetching a conversation

`convo2dlg` takes a payload, so how you fetch it is separate. The payload is the JSON that ChatGPT's web app renders from: `GET /backend-api/conversation/<id>` for a conversation you own, or `GET /backend-api/share/<id>` for a share link. Cloudflare blocks non-browser clients (a plain `httpx.get` gets a challenge page, public link or not), so the request has to run inside a logged-in browser. `url2convo` does that through a fastcdp `Page`: it runs the `fetch` in the page, so the call carries the session, and `_api_path` picks the endpoint from the URL form. A share link needs no auth token, only the browser context; a private or live conversation needs both.

You don't have to use `url2convo`. Anything that yields the same JSON feeds `convo2dlg`: a conversation from the Settings data export, or the one-line `fetch` below pasted into the browser's devtools console and saved to a file.

In [ ]:
#| export
_re_convo = re.compile(r'([0-9a-f]{8}(?:-[0-9a-f]{4}){3}-[0-9a-f]{12})')

def _api_path(url):
    "The backend-api path for a conversation or share `url`"
    return ('share/' if '/share/' in url else 'conversation/')+_re_convo.search(url).group(1)

async def url2convo(
    url, # Conversation or share URL (or bare conversation id)
    page, # A fastcdp `Page` in a logged-in browser
):
    "The conversation payload for `url`, fetched with the login held by `page`"
    if not await page.eval(r"location.host=='chatgpt.com'"): await page.goto('https://chatgpt.com')
    return await page.eval(f"""fetch('/api/auth/session').then(r=>r.json()).then(j=>
        fetch('/backend-api/{_api_path(url)}', {{headers:{{Authorization:'Bearer '+j.accessToken}}}}).then(r=>r.json()))""")

In [ ]:
test_eq(_api_path('https://chatgpt.com/c/6a771111-4324-83eb-8ba8-780911111111'), 'conversation/6a771111-4324-83eb-8ba8-780911111111')
test_eq(_api_path('https://chatgpt.com/share/e/6a795555-26a0-83eb-bf59-84d255555555?x=1'), 'share/6a795555-26a0-83eb-bf59-84d255555555')
test_eq(_api_path('6a771111-4324-83eb-8ba8-780911111111'), 'conversation/6a771111-4324-83eb-8ba8-780911111111')

In [ ]:
#| eval: false
# Convenience: fetch through a fastcdp browser tab that holds your ChatGPT login.
from fastcdp.skill import ExtCDP
cdp  = await ExtCDP.listen()
page = await cdp.new_page()
dlg  = convo2dlg(await url2convo('https://chatgpt.com/c/<conversation-id>', page))

# Manual alternative, no fastcdp: paste one of these into the browser devtools console,
# then save the result as a .json and pass its path to convo2dlg.
#   private/live: fetch('/api/auth/session').then(r=>r.json()).then(j=>
#                   fetch('/backend-api/conversation/<id>',{headers:{Authorization:'Bearer '+j.accessToken}})).then(r=>r.json())
#   public share: fetch('/backend-api/share/<share-id>').then(r=>r.json())

## Share vs conversation payloads

The two endpoints serve the same conversation differently, and the differences are worth knowing (all observed live on one conversation fetched both ways):

- The conversation endpoint returns the `mapping` tree pruned to what the UI shows its owner: on our test conversation, 125 nodes. The share endpoint returned 202 nodes for the *same* conversation: the extras are hidden `system` messages (flagged `is_visually_hidden_from_conversation`), `model_editable_context` snapshots (custom instructions/memory state), and memory-write traffic.
- The share payload's extra fields exist because a share link renders a standalone page: `linear_conversation` is the active path pre-walked (no client tree walk), `og_title`/`og_description` feed link previews, and `is_public`, `continue_conversation_url`, and `backing_conversation_id` describe the share itself. Share ids and conversation ids are distinct namespaces—the URL form picks the endpoint.
- Neither endpoint returns real `web.run` outputs: tool results read "The output of this plugin was redacted." What the UI shows as citations lives in each assistant message's `metadata.content_references`, keyed by `matched_text` (private-use glyph markers embedded in the reply text) with resolved title+url items—which is why `_splice_refs` can recover the links.
- Auth: conversation URLs need the owner's login; workspace `/share/e/` links need a workspace login too (anonymous requests get 403). Public `/share/` links may serve without auth.

Both wrappers converge after `convo_nodes`+`nodes2chat`: converting the same conversation from either source yields identical prompts, differing only in how much bookkeeping rides along as raws.

In [ ]:
#| eval: false
c = await url2convo('https://chatgpt.com/c/6a771111-4324-83eb-8ba8-780911111111', page)
s = await url2convo('https://chatgpt.com/share/e/6a795555-26a0-83eb-bf59-84d255555555', page)
cp = lambda x: [m.content for m in convo2dlg(x).messages if m.msg_type=='prompt']
test_eq(cp(c), cp(s))

## Export -

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()